In [1]:
import scanpy as sc
import os

In [2]:
adata_ref = sc.read_h5ad("../data/snrna_ref/MTG_snRNAseq.h5ad")
print(adata_ref)
print(adata_ref.obs.columns.tolist())

AnnData object with n_obs × n_vars = 137303 × 36601
    obs: 'sample_name', 'donor_sex_label', 'external_donor_name_label', 'species_label', 'age_label', 'region_label', 'cortical_layer_label', 'full_genotype_label', 'QCpass', 'cluster_label', 'cluster_confidence', 'subclass_label', 'subclass_confidence', 'class_label', 'class_confidence', 'GA_QCpass', 'GA_cluster_label', 'GA_subclass_label', 'GA_neighborhood_label', 'CA_QCpass', 'CA_cluster_label', 'CA_subclass_label', 'CA_neighborhood_label', 'cluster_color', 'cluster_order', 'subclass_color', 'subclass_order', 'class_color', 'class_order', 'GA_cluster_color', 'GA_cluster_order', 'GA_subclass_color', 'GA_subclass_order', 'CA_cluster_color', 'CA_cluster_order', 'CA_subclass_color', 'CA_subclass_order', 'cell_type_accession_label', 'n_genes', '_scvi_batch', '_scvi_labels'
    uns: '_scvi', 'cluster_label_colors', 'neighbors', 'subclass_label_colors', 'umap'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_categoricals', '_scvi_extra_continuo

In [3]:
print(adata_ref.obs['subclass_label'].value_counts())

subclass_label
L2/3 IT            28712
L5 IT              19580
L4 IT              16808
Sst                12393
Pvalb              10828
Vip                 9247
L6 IT               5966
Oligodendrocyte     5513
L6 IT Car3          4490
Lamp5               3356
L5/6 NP             3095
L6b                 2982
L6 CT               2683
Astrocyte           2590
Lamp5 Lhx6          2147
OPC                 2055
Sncg                1677
Microglia-PVM       1095
Chandelier           684
Pax6                 659
L5 ET                371
Sst Chodl            159
VLMC                 124
Endothelial           89
Name: count, dtype: int64


In [4]:
celltype_col = 'subclass_label'

def subsample_by_celltype(adata, col, n=500, seed=42):
    indices = []
    for ct in adata.obs[col].unique():
        ct_idx = adata.obs[adata.obs[col] == ct].index
        if len(ct_idx) > n:
            ct_idx = ct_idx.to_series().sample(n=n, random_state=seed).index
        indices.extend(ct_idx)
    return adata[indices].copy()

adata_ref_sub = subsample_by_celltype(adata_ref, celltype_col, n=500)
print(f"Subsampled: {adata_ref_sub.shape}")
print(adata_ref_sub.obs[celltype_col].value_counts())

Subsampled: (10743, 36601)
subclass_label
Astrocyte          500
Chandelier         500
L2/3 IT            500
L4 IT              500
L5/6 NP            500
L5 IT              500
L6 IT              500
L6 CT              500
Oligodendrocyte    500
Pax6               500
L6 IT Car3         500
L6b                500
Lamp5              500
Lamp5 Lhx6         500
Microglia-PVM      500
OPC                500
Sst                500
Vip                500
Pvalb              500
Sncg               500
L5 ET              371
Sst Chodl          159
VLMC               124
Endothelial         89
Name: count, dtype: int64


In [5]:
print(adata_ref_sub.X[:5, :5].toarray() if hasattr(adata_ref_sub.X, 'toarray') else adata_ref_sub.X[:5, :5])
print(f"max value: {adata_ref_sub.X.max()}")  # 如果 max > 100，大概率是 raw counts

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
max value: 6490.0


In [6]:
sc.pp.normalize_total(adata_ref_sub, target_sum=1e4)
sc.pp.log1p(adata_ref_sub)
print(f"After normalization, max value: {adata_ref_sub.X.max():.2f}")

After normalization, max value: 7.56


In [7]:
adata_ref_sub.write("../data/snrna_ref/MTG_ref_subsampled.h5ad")